# Maternal Health Risk Prediction

## K-Nearest Neighbors (KNN) Model

#### Import Required Libraries

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold
)

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

import pickle
import os

print("Starting KNN script...")

Starting KNN script...


#### Load Data

In [2]:
data = pd.read_csv("../Dataset/Maternal Health Risk Data Set.csv")
print("Data loaded. Shape:", data.shape)

data = data.copy()

Data loaded. Shape: (1014, 7)


#### Data Information

In [3]:
print("="*60)
print("Dataset Information")
print("="*60)

display(data.info())

print()

display(data.describe())

print()

display(data.describe(include="object"))

Dataset Information
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1014 entries, 0 to 1013
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Age          1014 non-null   int64  
 1   SystolicBP   1014 non-null   int64  
 2   DiastolicBP  1014 non-null   int64  
 3   BS           1014 non-null   float64
 4   BodyTemp     1014 non-null   float64
 5   HeartRate    1014 non-null   int64  
 6   RiskLevel    1014 non-null   object 
dtypes: float64(2), int64(4), object(1)
memory usage: 55.6+ KB


None

,Age,SystolicBP,DiastolicBP,BS,BodyTemp,HeartRate
count,1014.000000,1014.000000,1014.000000,1014.000000,1014.000000,1014.000000
mean,29.871795,113.198225,76.460552,8.725986,98.665089,74.301775
std,13.474386,18.403913,13.885796,3.293532,1.371384,8.088702
min,10.000000,70.000000,49.000000,6.000000,98.000000,7.000000
25%,19.000000,100.000000,65.000000,6.900000,98.000000,70.000000
50%,26.000000,120.000000,80.000000,7.500000,98.000000,76.000000
75%,39.000000,120.000000,90.000000,8.000000,98.000000,80.000000
max,70.000000,160.000000,100.000000,19.000000,103.000000,90.000000


,RiskLevel
count,1014
unique,3
top,low risk
freq,406


#### Feature Engineering

In [4]:
if {"SystolicBP", "DiastolicBP"}.issubset(data.columns):

    data["PulsePressure"] = (
        data["SystolicBP"] -
        data["DiastolicBP"]
    )

    eps = 1e-3

    data["BPRatio"] = (
        data["SystolicBP"] /
        (data["DiastolicBP"] + eps)
    )

if {"BS", "HeartRate"}.issubset(data.columns):

    data["BS_HR_Interaction"] = (
        data["BS"] *
        data["HeartRate"]
    )

if {"Age", "SystolicBP"}.issubset(data.columns):

    data["Age_SystolicBP"] = (
        data["Age"] *
        data["SystolicBP"]
    )

TARGET_COL = "RiskLevel"

X = data.drop(columns=[TARGET_COL])

y = data[TARGET_COL]

print("Features shape:", X.shape)

print()

print("Target distribution:\n")

print(y.value_counts())

Features shape: (1014, 10)

Target distribution:

RiskLevel
low risk     406
mid risk     336
high risk    272
Name: count, dtype: int64


#### Train/Test Split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    stratify=y,

    random_state=42

)

print("Train set:", X_train.shape)

print("Test set :", X_test.shape)

Train set: (811, 10)
Test set : (203, 10)


#### Pipeline (Scaler + KNN)

In [6]:
knn_pipe = Pipeline(

    [

        (

            "scaler",

            StandardScaler()

        ),

        (

            "knn",

            KNeighborsClassifier()

        )

    ]

)

print("Pipeline Created Successfully")

Pipeline Created Successfully


#### Hyperparameter Grid

In [7]:
param_grid = {

    "knn__n_neighbors": list(range(1, 31)),

    "knn__weights": [

        "uniform",

        "distance"

    ],

    "knn__metric": [

        "euclidean",

        "manhattan"

    ]

}

cv_strategy = StratifiedKFold(

    n_splits=5,

    shuffle=True,

    random_state=42

)

print("Hyperparameter Grid Ready")

Hyperparameter Grid Ready


#### Grid Search

In [8]:
print("Running GridSearchCV...")

grid_search = GridSearchCV(

    estimator=knn_pipe,

    param_grid=param_grid,

    cv=cv_strategy,

    scoring="accuracy",

    n_jobs=-1,

    verbose=2

)

grid_search.fit(

    X_train,

    y_train

)

print()

print("="*60)

print("Best CV Accuracy:", grid_search.best_score_)

print("Best Parameters:", grid_search.best_params_)

Running GridSearchCV...
Fitting 5 folds for each of 120 candidates, totalling 600 fits

Best CV Accuracy: 0.802696356888586
Best Parameters: {'knn__metric': 'manhattan', 'knn__n_neighbors': 2, 'knn__weights': 'distance'}


#### Evaluate Best KNN Model

In [9]:
try:

    best_model = grid_search.best_estimator_

    print("Training best model on full training data...")

    best_model.fit(
        X_train,
        y_train
    )

    y_pred = best_model.predict(X_test)

    acc = accuracy_score(
        y_test,
        y_pred
    )

    cm = confusion_matrix(
        y_test,
        y_pred
    )

    report = classification_report(
        y_test,
        y_pred
    )

    print("\n===== FINAL TEST PERFORMANCE (BEST KNN) =====")

    print("Test Accuracy:", acc)

    print("\nConfusion Matrix:\n")

    print(cm)

    print("\nClassification Report:\n")

    print(report)

except Exception as e:

    print("Error occurred:", repr(e))

print("Script finished.")

Training best model on full training data...

===== FINAL TEST PERFORMANCE (BEST KNN) =====
Test Accuracy: 0.8423645320197044

Confusion Matrix:

[[52  2  1]
 [ 3 66 12]
 [ 2 12 53]]

Classification Report:

              precision    recall  f1-score   support

   high risk       0.91      0.95      0.93        55
    low risk       0.82      0.81      0.82        81
    mid risk       0.80      0.79      0.80        67

    accuracy                           0.84       203
   macro avg       0.85      0.85      0.85       203
weighted avg       0.84      0.84      0.84       203

Script finished.


#### Save KNN Model

In [10]:
print("=" * 70)
print("Saving KNN Model")
print("=" * 70)

os.makedirs(
    "../Model",
    exist_ok=True
)

MODEL_PATH = "../Model/knn.pkl"

with open(
    MODEL_PATH,
    "wb"
) as file:

    pickle.dump(
        best_model,
        file
    )

print("Model Saved Successfully")
print("Location :", MODEL_PATH)

Saving KNN Model
Model Saved Successfully
Location : ../Model/knn.pkl
